# 03 - Comparing MOM6 output with the GLORYS12 reanalysis

Notebook 3 of 3. We put the regional MOM6 run side by side with the GLORYS12 daily-mean
reanalysis it was initialised and forced from, at the surface and in T-S space.

## Setup

History files are opened directly in the cells below; only the scientific operations are wrapped in functions.

In [ ]:
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}
import glob
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean.cm as cmo
import matplotlib.ticker as mticker
import xgcm

from mom6_tools.wright_eos import wright_eos, alpha_wright_eos, beta_wright_eos
from mom6_tools.jobqueue import get_cluster

xr.set_options(keep_attrs=True)

FIGDIR = Path("diagnostic_figures")
FIGDIR.mkdir(exist_ok=True)

### Start a Dask Cluster/Client
Dask is a parallel computation library for Python that lets us initialize multiple workers to act as parallel processors. There are two options for creating a dask cluster:
- PBSCluster: This submits new PBS jobs for each worker, running separate jobs on Casper/Derecho for each worker.
- LocalCluster: If you are running this notebook on an existing PBS job with enough resources, it can access those resources directly.

In [ ]:
DASK = "PBSCluster" # or LocalCluster (works with SLURMCluster on other systems)

N_WORKERS = 4

ACCOUNT = "P93300012" # charge worker jobs here, defaults to PROJECT and PBS_ACCOUNT env variables in bash
QUEUE = "casper" # or "derecho"
INTERFACE = 'ext' # or "ib0" for derecho

# Following are resources for each worker
# Total resources are N_WORKERS * resource (e.g. N_WORKERS * MEMORY).
MEMORY = "20GB"
CORES = 4
PROCESSES = 1
WALLTIME = "03:00:00"

success, client, cluster = get_cluster(nw=N_WORKERS, cluster_class=DASK, account=ACCOUNT, queue=QUEUE, walltime=WALLTIME, memory=MEMORY, processes=PROCESSES, cores = CORES, interface=INTERFACE, log_directory="logs",)

In [ ]:
client

## Case Selection

In [ ]:
# --- Case configuration ---
CASE = dict(
        casename="small_alaska",
        hist="/glade/derecho/scratch/manishrv/archive/small_alaska/ocn/hist",
        input_dir="/glade/u/home/manishrv/scratch/croc_input/small_alaska/ocn",
        date_range=["2020-01-01", "2020-01-08"],
        length="demo" # or "full", if you have more than 1 month of output. Otherwise native and z files will be filled values
    )

HIST = CASE["hist"]
INPUT_DIR = CASE["input_dir"]
DATE_RANGE = CASE["date_range"]
FILE_PATTERN = f"{HIST}/{CASE['casename']}.mom6"
MONTH = DATE_RANGE[0][:7]   # the monthly native/z file DATE_RANGE starts in

## Other Paths and Settings
GLORYS_ROOT = "/gdex/data/d010049"

# MOM6/FMS counts days with proleptic-Gregorian rules but labels the axis
# `calendar = "gregorian"`, which CF defines as the mixed Julian/Gregorian
# calendar. Open history files with decode_times=False and pass them to decode_mom6_time().
# If you are running with 'noleap', use `MOM6_CALENDAR = "noleap"`
MOM6_CALENDAR = "proleptic_gregorian"

print(f"{CASE['casename']}: {FILE_PATTERN}.h.*.nc")

In [ ]:
# --- The cost knob for this notebook: edit this ---
# GLORYS12 daily means are GLOBAL files of ~1.3 GB each. The window starts at
# the beginning of CASE["date_range"]; replace that with any date inside the run.
start_date = pd.to_datetime(DATE_RANGE[0])
end_date = pd.to_datetime(DATE_RANGE[1])

DATES = [str(d)[:10] for d in pd.date_range(start = start_date, end = end_date, freq="D")]
print(f"{CASE['casename']}: GLORYS {DATES[0]} -> {DATES[-1]} ({len(DATES)} daily file(s))")


### Relevant Files
- STATIC - `casename.mom6.h.static.nc`: contains real lat/lon coords, grid information, coriolis parameter
- GEOM - `casename.mom6.h.ocean_geometry.nc`: a lot of similar info to STATIC, includes bathymetry

The provided geolat/geolon coords give physical lat/lon adjusted across the grid for the most accurate plotting and geolocating. Default xh/yh coords are the nominal lat/lon which are best used as indices and temporary coords before rigorous scientific plotting/analysis.

In [ ]:
# --- Grid files ---
# The static file carries the land masks, cell areas and Coriolis parameter.
STATIC = xr.open_dataset(f"{FILE_PATTERN}.h.static.nc", decode_times=False).squeeze(drop=True)

# Rename geometry coords to the history-file names.
GEOM = (xr.open_dataset(f"{FILE_PATTERN}.h.ocean_geometry.nc")
          .rename_dims({"lath": "yh", "lonh": "xh", "latq": "yq", "lonq": "xq"})
          .drop_vars(["lath", "lonh", "latq", "lonq"]))

# Attach these to any history DATASET with `ds = ds.assign_coords(COORDS)`
# This streamlines plotting and xgcm grid operations, which expect the geometry coords to be present.
COORDS = {name: GEOM[name] for name in
          ("geolon", "geolat", "geolonu", "geolatu",
           "geolonv", "geolatv", "geolonb", "geolatb")}

# Example: STATIC now stores geolon/geolat as data variables (psst it already had them)
STATIC = STATIC.assign_coords(COORDS)

print(dict(STATIC.sizes))
print("lon {:.2f} .. {:.2f}   lat {:.2f} .. {:.2f}".format(
    float(GEOM.geolon.min()), float(GEOM.geolon.max()),
    float(GEOM.geolat.min()), float(GEOM.geolat.max())))

### Formatting Plots
For efficiency in these notebooks, we provide some standard plot settings and information. This lets us quickly use the implicit plotting capapbilities of xarray (i.e. dataarray.plot(kwargs**)).

In [ ]:
# --- Standard plotting keywords ---
# Everything is plotted with xarray's own `.plot()`; these dicts just supply the
# coordinates and the transform so everything is standardized and pretty!

SUBPLOT = {"projection": ccrs.Robinson(central_longitude=float(GEOM.geolon.mean()))}
LAND = cfeature.LAND.with_scale("50m")

_aspect = float(GEOM.geolat.max() - GEOM.geolat.min()) / float(GEOM.geolon.max() - GEOM.geolon.min())
PANEL = np.array([18.0, round(18.0 * _aspect)])

# Lat lon gridlines for all plots. Use `ax.gridlines(**GRID)` to apply.
nticks = 7
GRID = dict(
      draw_labels=["bottom", "left"],
      xlocs=mticker.MultipleLocator(np.round(np.abs((GEOM.geolon.max().values - GEOM.geolon.min().values) / nticks), 1)),
      ylocs=mticker.MultipleLocator(np.round(np.abs((GEOM.geolat.max().values - GEOM.geolat.min().values) / nticks), 1)),
      linewidth=0.5, color="0.5", alpha=0.5,
  )

# Fast map for plotting MOM6 fields. Use `ds.plot(**MAP[grid])` where grid is one of "h", "u", "v", or "q".
# h - tracers, q - corners, u - u-velocity, v - v-velocity
MAP = {
    "h": dict(x="geolon",  y="geolat",  transform=ccrs.PlateCarree()),
    "u": dict(x="geolonu", y="geolatu", transform=ccrs.PlateCarree()),
    "v": dict(x="geolonv", y="geolatv", transform=ccrs.PlateCarree()),
    "q": dict(x="geolonb", y="geolatb", transform=ccrs.PlateCarree()),
}

# add-ons to spread over a `.plot()` call
DIFF = dict(cmap="cmo.balance", center=0, robust=True)   # diverging / difference
SECTION = dict(yincrease=False)                          # depth down the y axis

# SECTION only sets the axis direction, so it combines with anything:
#     da.plot(ax=ax, **SECTION, **DIFF)              a difference section
#     da.plot(ax=ax, **SECTION, robust=True)         clip the colour range
# Add robust=True deliberately -- on a shallow shelf it can clip a real deep
# temperature core out of the picture.

### xgcm grid and metrics
We use the xgcm for efficient handling of the Arakawa C-grid geometry. We provide information about which corrdinates correspond to tracer (center) points velocity (face) points. We also provide grid metrics from the geometry file into the xgcm grid and the relevant datasets.
Metrics give grid cell spacing and areas for different operations (e.g. area-weighted interpolation and grid size based difference).

The coordinates and metrics in the xgcm grid need to be present in the history file.

In [ ]:
# --- xgcm grid and derived quantities ---

XGCM_COORDS = {
    "X": {"center": "xh", "outer": "xq"},
    "Y": {"center": "yh", "outer": "yq"},
    "Z": {"center": "zl", "outer": "zi"},
}

def make_grid(ds, geom, with_metrics=True, xgcm_coords=XGCM_COORDS):
    """Build an xgcm.Grid for a MOM6 regional dataset, as (grid, ds_with_metrics).

    Metrics come from the ocean_geometry file `geom` so that grid.diff /
    grid.interp / grid.derivative / grid.integrate know the cell spacings.
    """
    coords = {ax: pos for ax, pos in xgcm_coords.items() if pos["center"] in ds.dims} # flexible for 3D/2D data
    obj, metrics = ds, None
    
    if with_metrics:
        for v in ("dxCu", "dyCu", "dxCv", "dyCv", "dxT", "dyT",
                  "dxBu", "dyBu", "Ah", "Aq"):
            if v in geom and set(geom[v].dims) <= set(ds.dims):
                obj = obj.assign_coords({v: geom[v]})
        metrics = {
            ("X",): [v for v in ("dxT", "dxCu", "dxCv", "dxBu") if v in obj.coords],
            ("Y",): [v for v in ("dyT", "dyCu", "dyCv", "dyBu") if v in obj.coords],
            ("X", "Y"): [v for v in ("Ah", "Aq") if v in obj.coords],
        }
        metrics = {k: v for k, v in metrics.items() if v}
        
    grid = xgcm.Grid(obj, coords=coords, metrics=metrics, padding="extend",
                autoparse_metadata=False)
    
    return grid, obj

def decode_mom6_time(ds, calendar="proleptic_gregorian"):
    """Decode MOM6 time axes, correcting the calendar that MOM6 mislabels.
    """
    ds = ds.copy()
    for v in ds.variables:
        attrs = dict(ds[v].attrs)
        if "since" in str(attrs.get("units", "")):
            attrs["calendar"] = calendar
            ds[v].attrs = attrs
    return xr.decode_cf(ds)

def relative_vorticity(grid, ds, u="uo", v="vo"):
    """Relative vorticity at the corner (q) point [s-1], the MOM6 discretisation:

        zeta = ( d/dx (v * dyCv) - d/dy (u * dxCu) ) / Aq
    """
    dvdx = grid.diff(ds[v] * ds["dyCv"], "X", padding="fill", fill_value=0.0)
    dudy = grid.diff(ds[u] * ds["dxCu"], "Y", padding="fill", fill_value=0.0)
    zeta = (dvdx - dudy) / ds["Aq"]
    zeta.name = "zeta"
    zeta.attrs = {"long_name": "Relative vorticity", "units": "s-1"}
    return zeta


def layer_depths(e):
    """Layer-centre depths (positive down, m) from interface heights `e`."""
    z = -0.5 * (e.isel(zi=slice(None, -1)).values + e.isel(zi=slice(1, None)).values)
    dims = [d if d != "zi" else "zl" for d in e.dims]
    coords = {k: v for k, v in e.coords.items() if "zi" not in v.dims}
    out = xr.DataArray(z, dims=dims, coords=coords, name="z_center")
    out.attrs = {"long_name": "Layer centre depth", "units": "m", "positive": "down"}
    return out

def density(T, S, p=0.0):
    """In-situ density [kg m-3] from the MOM6 Wright (1997) EOS.

    T in degC, S in psu, p in Pa.  T and S must be xarray objects (wrap a T/S
    mesh in xr.DataArray when contouring sigma0).  Uses mom6_tools.wright_eos.
    """
    rho = xr.apply_ufunc(wright_eos, T, S, p, dask="allowed", keep_attrs=False)
    rho.name = "rho"
    rho.attrs = {"long_name": "In-situ density (Wright 1997 EOS)", "units": "kg m-3"}
    return rho

def sigma0(T, S):
    """Potential density anomaly referenced to 0 dbar [kg m-3]."""
    s = density(T, S, 0.0) - 1000.0
    s.name = "sigma0"
    s.attrs = {"long_name": "Potential density anomaly (sigma-0)", "units": "kg m-3"}
    return s


In [ ]:
# --- GLORYS12 access ---
# One global daily-mean file per day, ~1.3 GB each, read straight off /gdex.
GLORYS_VARS = ("thetao", "so", "zos")


def glorys_files(dates, root):
    """The GLORYS12 daily-mean file for each 'YYYY-MM-DD' in `dates`, under `root`."""
    files = []
    for d in dates:
        y, m, dd = d.split("-")
        hits = sorted(glob.glob(f"{root}/{y}/*_{y}{m}{dd}_*.nc"))
        if not hits:
            raise FileNotFoundError(f"no GLORYS file for {d} under {root}/{y}")
        files.append(hits[0])
    return files


def open_glorys(dates, geom, root, pad=0.25):
    """Open GLORYS12 daily means over the model's lon/lat box, surface level only.

    `open_mfdataset` reads the coordinates but leaves the fields as dask arrays,
    so the `.sel` below is what decides how much actually comes off disk -- no
    per-file preprocessing needed.  Longitudes are put on the same convention as
    the model grid, which is what makes a dateline-straddling domain work.
    """
    ds = xr.open_mfdataset(glorys_files(dates, root), combine="by_coords", chunks={"time": 1})
    ds = ds[list(GLORYS_VARS)].isel(depth=0, drop=True)

    if float(geom.geolon.max()) > 180.0:            # model grid runs 0..360
        ds = ds.assign_coords(longitude=ds["longitude"] % 360).sortby("longitude")

    return ds.sel(
        longitude=slice(float(geom.geolon.min()) - pad, float(geom.geolon.max()) + pad),
        latitude=slice(float(geom.geolat.min()) - pad, float(geom.geolat.max()) + pad),
    )


In [ ]:
# --- Comparison helpers: regridding and error statistics ---

def to_model_grid(obj, geom):
    """Sample a GLORYS field (1-D latitude/longitude) onto the model h points.

    `geolon`/`geolat` are 2-D with dims (yh, xh), so passing them as the
    indexers makes `interp` do *pointwise* (advanced) interpolation -- one
    bilinear sample per model cell centre, rather than an outer product.  This
    is how we regrid onto a curvilinear grid without xesmf.
    """
    return (obj.interp(longitude=geom.geolon, latitude=geom.geolat)
               .drop_vars(["longitude", "latitude"], errors="ignore")
               .assign_coords(geolon=geom.geolon, geolat=geom.geolat))


def common_mask(a, b):
    """Restrict a model/observation pair to the cells where both are valid."""
    both = a.notnull() & b.notnull()
    return a.where(both), b.where(both)


def area_stats(mod, obs, area):
    """Area-weighted bias, RMSE and pattern (centred) correlation of `mod` vs `obs`.

    `mod` and `obs` have already been through `common_mask`, so xarray's
    `.weighted()` reductions skip exactly the same cells in both.
    """
    w = area.fillna(0.0)                       # .weighted() rejects NaN weights
    mbar, obar = mod.weighted(w).mean(), obs.weighted(w).mean()
    ma, oa = mod - mbar, obs - obar
    r = ((ma * oa).weighted(w).mean()
         / np.sqrt((ma ** 2).weighted(w).mean() * (oa ** 2).weighted(w).mean()))
    return {"n_cells": int(mod.notnull().sum()),
            "model_mean": float(mbar), "glorys_mean": float(obar),
            "bias": float(mbar - obar),
            "rmse": float(np.sqrt(((mod - obs) ** 2).weighted(w).mean())),
            "pattern_r": float(r)}


def ts_axis(objs, name, dim, n=60):
    """A 1-D axis spanning the combined range of `name` over the datasets `objs`.

    Two of these, on different dims, broadcast against each other into the T-S
    mesh that `sigma0` is contoured on.
    """
    v = np.linspace(min(float(o[name].min()) for o in objs),
                    max(float(o[name].max()) for o in objs), n)
    return xr.DataArray(v, dims=dim, coords={dim: v})


## 1. Which GLORYS files are we about to read?

`open_glorys` derives the model's lon/lat box from `GEOM` and subsets the lazily-opened files, so
only the surface level over a small window of each global file is ever read -- but run time still
scales with `N_DAYS`.


In [ ]:
from mom6_tools.regional_m6toolbox import glorys_files

files = glorys_files(DATES, GLORYS_ROOT)
pd.DataFrame({
    "date": DATES,
    "file": [os.path.basename(f) for f in files],
    "size_GB": [round(os.path.getsize(f) / 1e9, 2) for f in files],
})


## 2. The model grid and the GLORYS subset

Open the surface stream. The time axis needs the FMS calendar offset, and the 2-D coordinates
come from `COORDS`.

In [ ]:
from mom6_tools.regional_m6toolbox import decode_mom6_time

# --- Surface stream (daily records) ---
sfc = xr.open_mfdataset(f"{FILE_PATTERN}.h.sfc.*.nc", decode_times=False,
                      chunks={"time": 1})
sfc = decode_mom6_time(sfc, calendar=MOM6_CALENDAR).sel(time=slice(*DATE_RANGE))     # MOM6 mislabels its calendar; fix then decode
sfc = sfc.assign_coords(COORDS)

print(dict(sfc.sizes))
print(str(sfc.time.values[0])[:10], "->", str(sfc.time.values[-1])[:10],
      f"({sfc.sizes['time']} records)")

# The sea-surface-height diagnostic is named `zos` in one case and `SSH` in the other.
SSH_VAR = next(v for v in ("zos", "SSH", "ssh") if v in sfc)
print("model SSH variable:", SSH_VAR)

In [ ]:
from mom6_tools.regional_m6toolbox import open_glorys

gl_ds = open_glorys(DATES, GEOM, GLORYS_ROOT)

print("GLORYS sizes :", dict(gl_ds.sizes))
print("GLORYS lon   : %.2f .. %.2f" % (float(gl_ds.longitude.min()), float(gl_ds.longitude.max())))
print("model  lon   : %.2f .. %.2f" % (float(GEOM.geolon.min()), float(GEOM.geolon.max())))
print("GLORYS lat   : %.2f .. %.2f" % (float(gl_ds.latitude.min()), float(gl_ds.latitude.max())))
print("model  lat   : %.2f .. %.2f" % (float(GEOM.geolat.min()), float(GEOM.geolat.max())))
print("GLORYS times :", [str(t)[:10] for t in gl_ds.time.values])


## 3. Regridding GLORYS onto the model grid

`xesmf` is not available here, so `to_model_grid` uses `xr.DataArray.interp` with the 2-D
`geolon`/`geolat` indexers built above -- pointwise (advanced) interpolation onto the curvilinear grid.

In [ ]:
from mom6_tools.regional_m6toolbox import to_model_grid

DATE = DATES[len(DATES) // 2]          # the middle day of the window, for the surface maps
mod_sfc = sfc.sel(time=DATE, method="nearest")
obs_sfc = to_model_grid(gl_ds.sel(time=DATE, method="nearest").load(), GEOM)

print("comparison date :", str(mod_sfc.time.values)[:10])
print("model  tos      :", dict(mod_sfc["tos"].sizes))
print("GLORYS thetao   :", dict(obs_sfc["thetao"].sizes), "<- now on the model grid")

Plot the model footprint over the raw GLORYS subset once, to confirm the window is the right one.

In [ ]:
fig, ax = plt.subplots(figsize=(PANEL[0], PANEL[1] + 0.4), subplot_kw=SUBPLOT)
gl_ds["thetao"].isel(time=0).load().plot(
    ax=ax, x="longitude", y="latitude", transform=ccrs.PlateCarree(), cmap="cmo.thermal")

lo, la = GEOM.geolon.values, GEOM.geolat.values
edge_lon = np.concatenate([lo[0, :], lo[:, -1], lo[-1, ::-1], lo[::-1, 0]])
edge_lat = np.concatenate([la[0, :], la[:, -1], la[-1, ::-1], la[::-1, 0]])
ax.plot(edge_lon, edge_lat, transform=ccrs.PlateCarree(), color="crimson", lw=1.8,
        label="MOM6 domain")

ax.coastlines("50m")
ax.add_feature(LAND, facecolor="0.85")
gl = ax.gridlines(**GRID)
ax.set_extent([float(GEOM.geolon.min()) - 2, float(GEOM.geolon.max()) + 2,
               float(GEOM.geolat.min()) - 1, float(GEOM.geolat.max()) + 1], crs=ccrs.PlateCarree())
ax.legend(loc="lower left", fontsize=8)
fig.suptitle("GLORYS12 SST subset with the model footprint")

## 4. Surface comparison

Model, GLORYS and their difference for SST, SSS and SSH. MOM6 and GLORYS use **different SSH
reference levels**, so the domain mean is removed from each before differencing.

In [ ]:
from mom6_tools.regional_m6toolbox import common_mask

AREA = STATIC["areacello"]

SURFACE = [
    ("SST", "degC", mod_sfc["tos"],    obs_sfc["thetao"], "cmo.thermal", False),
    ("SSS", "psu",  mod_sfc["sos"],    obs_sfc["so"],     "cmo.haline",  False),
    ("SSH", "m",    mod_sfc[SSH_VAR],  obs_sfc["zos"],    "cmo.balance", False),
]

surface_pairs = {}
for name, units, mod, obs, cmap, demean in SURFACE:
    mod, obs = common_mask(mod.where(STATIC.wet), obs)
    if demean:                      # different reference levels -> compare anomalies
        w = AREA.fillna(0.0)
        off_mod, off_obs = float(mod.weighted(w).mean()), float(obs.weighted(w).mean())
        print(f"{name}: removing domain means -- MOM6 {off_mod:+.3f} {units}, "
              f"GLORYS {off_obs:+.3f} {units} (different reference levels)")
        mod, obs = mod - off_mod, obs - off_obs
    surface_pairs[name] = (units, mod, obs)

    diff = mod - obs
    vmin = min(float(mod.min()), float(obs.min()))   # shared scale for the two data panels
    vmax = max(float(mod.max()), float(obs.max())) 

    cbar = {"label": f"{name} [{units}]"}
    fig, axs = plt.subplots(3, 1, figsize=(PANEL[0], 3*PANEL[1]), subplot_kw=SUBPLOT)
    for ax, (fld, lab, kw) in zip(axs, [
            (mod,  "MOM6",          dict(cmap=cmap, vmin=vmin, vmax=vmax, cbar_kwargs=cbar)),
            (obs,  "GLORYS12",      dict(cmap=cmap, vmin=vmin, vmax=vmax, cbar_kwargs=cbar)),
            (diff, "MOM6 - GLORYS", dict(**DIFF,  cbar_kwargs={"label": f"{name} difference [{units}]"}))]):
        if name == "SSH":  # SSH is a small number, so use a different scale for the difference
            kw = dict(cmap=cmo.balance, center=0, robust=True, cbar_kwargs={"label": f"{name} difference [{units}]"})
        fld.plot(ax=ax, **MAP["h"], **kw)
        ax.coastlines("50m")
        ax.add_feature(LAND, facecolor="0.85")
        ax.set_title(lab)
        g = ax.gridlines(**GRID)
    fig.suptitle(f"{CASE['casename']}  {name}  {str(mod_sfc.time.values)[:10]}")
    fig.savefig(FIGDIR / f"03_surface_{name}.png", dpi=120)


## 5. Error statistics

Area-weighted bias, RMSE and pattern correlation over every cell where both fields are valid.

In [ ]:
from mom6_tools.regional_m6toolbox import area_stats

table = pd.DataFrame({f"{name} [{units}]": area_stats(mod, obs, AREA)
                      for name, (units, mod, obs) in surface_pairs.items()}).T
table.round(4)

## 6. T-S properties

Surface water masses in T-S space over every day of the window, with sigma-0 contours from the
MOM6 Wright (1997) equation of state.

Both fields stay xarray objects the whole way: `THIN` subsamples them on the model grid, and
matplotlib flattens and skips the land NaNs on its own when they are handed to `scatter`.


In [ ]:
from mom6_tools.regional_m6toolbox import sigma0, to_model_grid, ts_axis

# The model and the GLORYS daily means are both stamped 12:00, so the time axes
# line up and `sel` picks the matching day out of each.
THIN = 8       # plot every THIN-th model cell in each direction

mod_ts = (xr.Dataset({"thetao": sfc["tos"], "so": sfc["sos"]})
            .sel(time=DATES, method="nearest").where(STATIC.wet))
obs_ts = to_model_grid(gl_ds[["thetao", "so"]].load(), GEOM)

thin = dict(yh=slice(None, None, THIN), xh=slice(None, None, THIN))
mod_pts, obs_pts = mod_ts.isel(**thin).load(), obs_ts.isel(**thin)
print(f"{mod_pts['thetao'].count().values:,} model points, "
      f"{obs_pts['thetao'].count().values:,} GLORYS points")

# sigma-0 background: two 1-D axes that broadcast against each other into a T-S mesh.
salt_ax = ts_axis([mod_pts, obs_pts], "so", "salinity")
temp_ax = ts_axis([mod_pts, obs_pts], "thetao", "temperature")
sig = sigma0(temp_ax, salt_ax)

fig, ax = plt.subplots(figsize=(6.2, 5.6))
cs = sig.plot.contour(ax=ax, x="salinity", y="temperature", levels=10,
                      colors="0.55", linewidths=0.7)
ax.clabel(cs, fontsize=7, fmt="%.1f")
ax.scatter(obs_pts["so"], obs_pts["thetao"], s=4, alpha=0.35, color="tab:orange", label="GLORYS12")
ax.scatter(mod_pts["so"], mod_pts["thetao"], s=4, alpha=0.35, color="tab:blue", label="MOM6")
ax.set_xlabel("Salinity [psu]")
ax.set_ylabel("Temperature [degC]")
ax.legend(markerscale=4, fontsize=8)
fig.suptitle(f"{CASE['casename']}  surface T-S, {DATES[0]} .. {DATES[-1]}  (contours: sigma-0)")
fig.tight_layout()
fig.savefig(FIGDIR / "03_ts_diagram.png", dpi=120)


## Where to go next

Raise `N_DAYS` for a longer window, or point `CASE` at a different run. Note that `CASE` is
assigned twice in the case-selection cell above: the second assignment is the one that takes
effect.


## Cleanup

In [ ]:
done = False # let's you use Run All without closing the cluster.
if done:
    client.close()
    cluster.close()